# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mvdu12/ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of analysis: one row = one content page (content_hash_id), with its structural
attributes from dim_content joined to its performance aggregated over one month from
fact_content_daily_performance.

Time window: one full month — month = '2026-03'
(report_date between 2026-03-01 and 2026-03-31)

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

FEATURES (knowable at the decision moment):
- From dim_content: word_count, char_count, content_type, main_intent,
  search_volume, competition, competition_level, cpc, backlinks, category_count
- From fact_content_daily_performance (aggregated over the month):
  avg_gsc_position, sum(gsc_impressions), sum(gsc_clicks), sum(ga4_sessions),
  engagement_rate = ga4_engaged_sessions / ga4_sessions, sum(sessions_ai)

LABEL / PROXY:
None required — this is unsupervised archetype clustering. One label-derived
column is added only in the leakage trap experiment (Section 3) to demonstrate
the risk, then removed.

CONTEXT (identifiers, not features):
client_hash_id, content_hash_id, month, is_published, is_deleted,
client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available

EXCLUDED (and why):
- fact_content_query_90d — different grain (query-level, not content-level);
  joining it would break the unit of analysis
- keyword_hash_id / url_hash_id — identifiers, not features
- any month other than 2026-03 when building features — avoids time leakage
- rows where is_deleted = TRUE — not part of this lane's population

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:
# --- Query 1: Grain check — one row really is what I said ---
q1 = con.sql(f"""
    SELECT content_hash_id, COUNT(*) AS n_rows
    FROM read_parquet('{rel}/dim_content.parquet')
    GROUP BY content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").fetchdf()
print("Duplicate content_hash_id rows in dim_content (should be empty):")
print(q1)

# --- Query 2: Slice row count + date span for month=2026-03 ---
q2 = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT content_hash_id) AS n_distinct_pages
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").fetchdf()
print("\nRow count and date span for month=2026-03:")
print(q2)

# --- Query 3: Availability — filter with IS TRUE, show how many rows survive ---
q3 = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").fetchdf()
print("\nAvailability check for month=2026-03:")
print(q3)

Duplicate content_hash_id rows in dim_content (should be empty):
Empty DataFrame
Columns: [content_hash_id, n_rows]
Index: []

Row count and date span for month=2026-03:
    n_rows   min_date   max_date  n_distinct_pages
0  9841378 2026-03-01 2026-03-31            331437

Availability check for month=2026-03:
   total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
0     9841378             3611061              413966               364347


# --- Five features (max), each with an "available when?" line ---

1. word_count — available at the decision moment because it is set when the page
   is published and does not depend on future traffic.

2. content_type — available at the decision moment because it is assigned at
   creation and is static metadata from dim_content.

3. avg_gsc_position (aggregated over 2026-03) — available at the decision moment
   because it reflects ranking observed during the window itself, not after it.

4. engagement_rate = ga4_engaged_sessions / ga4_sessions (aggregated over 2026-03)
   — available at the decision moment because it is computed from the same month
   being analyzed, not a later one.

5. backlinks — available at the decision moment because it is a point-in-time
   count from dim_content, not derived from the label window.

In [10]:
# --- The trap: add ONE label-derived column on purpose, then remove it ---

# Build the honest feature frame for month=2026-03 (features above)
# NOTE: filtered by gsc_data_available / ga4_data_available IS TRUE,
# matching what Query 3 proved is actually usable.
honest_frame = con.sql(f"""
    SELECT
        c.content_hash_id, c.word_count, c.content_type, c.backlinks,
        AVG(f.gsc_avg_position) AS avg_gsc_position,
        SUM(f.ga4_engaged_sessions) * 1.0 / NULLIF(SUM(f.ga4_sessions), 0) AS engagement_rate
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
        ON c.content_hash_id = f.content_hash_id
    WHERE c.is_deleted IS FALSE
      AND f.gsc_data_available IS TRUE
      AND f.ga4_data_available IS TRUE
    GROUP BY c.content_hash_id, c.word_count, c.content_type, c.backlinks
""").fetchdf()

# Leak: pull in NEXT month's sessions as a "feature" (this is really outcome data)
leaked = con.sql(f"""
    SELECT content_hash_id, SUM(ga4_sessions) AS next_month_sessions
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_hash_id
""").fetchdf()

leaky_frame = honest_frame.merge(leaked, on='content_hash_id', how='left')

# --- Honest quick score: built ONLY from same-month features ---
final_frame_preview = honest_frame.copy()
final_frame_preview['quick_score_honest'] = (
    final_frame_preview['engagement_rate'].rank(pct=True) +
    (1 - final_frame_preview['avg_gsc_position'].rank(pct=True))
) / 2

# --- Leaky quick score: same formula, but next_month_sessions sneaks in as a "feature" ---
leaky_frame['quick_score_leaky'] = (
    leaky_frame['engagement_rate'].rank(pct=True) +
    (1 - leaky_frame['avg_gsc_position'].rank(pct=True)) +
    leaky_frame['next_month_sessions'].rank(pct=True)
) / 3

print("Honest frame shape:", honest_frame.shape)
print("Leaky frame shape:", leaky_frame.shape)

print("\nCorrelation of HONEST quick_score with next_month_sessions (should be near 0):")
honest_check = leaky_frame[['quick_score_leaky', 'next_month_sessions']].copy()
honest_check['quick_score_honest'] = final_frame_preview['quick_score_honest']
print(honest_check[['quick_score_honest', 'next_month_sessions']].corr(method='spearman'))

print("\nCorrelation of LEAKY quick_score with next_month_sessions (watch it jump toward 1.0):")
print(leaky_frame[['quick_score_leaky', 'next_month_sessions']].corr(method='spearman'))

# Delete the leaked column and the leaky score, keep only the honest frame
final_frame = final_frame_preview.copy()
print("\nFinal honest feature frame:", final_frame.shape)

Honest frame shape: (63851, 6)
Leaky frame shape: (63851, 8)

Correlation of HONEST quick_score with next_month_sessions (should be near 0):
                     quick_score_honest  next_month_sessions
quick_score_honest             1.000000             0.226761
next_month_sessions            0.226761             1.000000

Correlation of LEAKY quick_score with next_month_sessions (watch it jump toward 1.0):
                     quick_score_leaky  next_month_sessions
quick_score_leaky             1.000000             0.743343
next_month_sessions           0.743343             1.000000

Final honest feature frame: (63851, 7)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. Unbalanced panel: dim_clients.gsc_data_start and ga4_data_start differ per
   client, so not every content_hash_id has the same history depth. A page with
   a later gsc_data_start will show fewer historical months than an older
   client's page, even if both are equally healthy — this is a coverage gap,
   not a performance signal.

2. Availability is per-row, not guaranteed: gsc_data_available and
   ga4_data_available can be FALSE even within an active client-month, so raw
   row counts overstate what's actually usable. Every volume claim must be
   filtered with these flags first.

3. Grain mismatch risk: fact_content_query_90d is a rolling 90-day query-level
   window (window_start/window_end), not month-aligned with
   fact_content_daily_performance, and its last30/prev30 columns overlap in
   time with the daily fact table. Joining it naively would double-count
   activity for the same days — this is why it is excluded from this lane's
   feature frame.

4. No causal claim: this slice can show which structural archetypes correlate
   with stronger or weaker engagement in a given month — it cannot say a
   specific content change caused a ranking or traffic change, since no
   experiment or before/after control exists here.

5. Single-month snapshot: features are built from month=2026-03 only, so
   seasonal or one-off events in that month could shape the archetype clusters
   in ways that would not generalize to other months.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.